# Подсчёт автомобилей на видео с помощью YOLOv8

В этом ноутбуке реализована программа для детекции, трекинга и подсчёта автомобилей на видео.

Используется:
- YOLOv8 для поиска автомобилей;
- BoT-SORT для трекинга объектов;
- OpenCV для обработки видео и визуализации.

Перед запуском положите входное видео рядом с ноутбуком и назовите его `video.mp4`.


## 1. Установка библиотек

Эту ячейку нужно выполнить один раз, если библиотеки ещё не установлены.


In [ ]:
%pip install ultralytics opencv-python


## 2. Импорт библиотек


In [ ]:
import cv2
from ultralytics import YOLO
from IPython.display import Video, display


## 3. Настройки проекта

Здесь можно поменять путь к видео, модель YOLO, координаты линии и параметры детекции.

Для моего видео была выбрана вертикальная линия, потому что автомобили в основном движутся слева направо и справа налево.


In [ ]:
VIDEO_PATH = "video.mp4"
MODEL_PATH = "yolov8s.pt"
OUTPUT_VIDEO = "result.mp4"

# Линия подсчёта: две точки (x, y)
LINE_POINT_1 = (1050, 300)
LINE_POINT_2 = (1050, 780)

# Направление подсчёта:
# "A_TO_B" — в одну сторону
# "B_TO_A" — в обратную сторону
# "BOTH"   — в обе стороны
COUNT_DIRECTION = "BOTH"

# Классы COCO:
# 2 = car, 3 = motorcycle, 5 = bus, 7 = truck
VEHICLE_CLASSES = [2, 3, 5, 7]

CONFIDENCE = 0.20
IMAGE_SIZE = 960


## 4. Вспомогательные функции

Функции ниже определяют, пересёк ли центр объекта линию подсчёта.


In [ ]:
def get_point_side(point, line_point_1, line_point_2):
    """
    Определяет сторону точки относительно линии.
    """
    x, y = point
    x1, y1 = line_point_1
    x2, y2 = line_point_2

    side = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)

    return side


def check_crossing(previous_center, current_center):
    """
    Проверяет, пересёк ли объект линию подсчёта.
    """
    if previous_center is None:
        return False

    previous_side = get_point_side(previous_center, LINE_POINT_1, LINE_POINT_2)
    current_side = get_point_side(current_center, LINE_POINT_1, LINE_POINT_2)

    # Если знак изменился, значит центр объекта пересёк линию
    if previous_side * current_side < 0:
        if COUNT_DIRECTION == "BOTH":
            return True

        if COUNT_DIRECTION == "A_TO_B":
            return previous_side > 0 and current_side < 0

        if COUNT_DIRECTION == "B_TO_A":
            return previous_side < 0 and current_side > 0

    return False


def draw_text_with_background(frame, text, position):
    """
    Рисует текст с чёрным фоном, чтобы счётчик было хорошо видно.
    """
    x, y = position

    cv2.rectangle(frame, (x - 10, y - 35), (x + 260, y + 10), (0, 0, 0), -1)

    cv2.putText(
        frame,
        text,
        (x, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )


## 5. Основной код обработки видео

На каждом кадре программа:
1. находит транспорт;
2. получает ID объекта;
3. вычисляет центр bounding box;
4. проверяет пересечение линии;
5. рисует рамки, центры, ID и счётчик;
6. сохраняет результат в `result.mp4`.


In [ ]:
model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Ошибка: не удалось открыть видео.")
    print("Проверьте, что файл video.mp4 лежит рядом с ноутбуком.")
else:
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps == 0:
        fps = 25

    cap.release()

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    output = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

    car_count = 0
    counted_ids = set()
    previous_centers = {}

    results = model.track(
        source=VIDEO_PATH,
        stream=True,
        persist=True,
        tracker="botsort.yaml",
        classes=VEHICLE_CLASSES,
        conf=CONFIDENCE,
        imgsz=IMAGE_SIZE,
        verbose=False
    )

    for result in results:
        frame = result.orig_img.copy()

        if result.boxes is not None:
            for box in result.boxes:
                if box.id is None:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                track_id = int(box.id[0].cpu().numpy())
                class_id = int(box.cls[0].cpu().numpy())
                confidence = float(box.conf[0].cpu().numpy())

                if class_id not in VEHICLE_CLASSES:
                    continue

                center_x = int((x1 + x2) / 2)
                center_y = int((y1 + y2) / 2)
                current_center = (center_x, center_y)

                previous_center = previous_centers.get(track_id)

                color = (0, 255, 0)

                if track_id not in counted_ids:
                    if check_crossing(previous_center, current_center):
                        car_count += 1
                        counted_ids.add(track_id)
                        color = (0, 0, 255)

                if track_id in counted_ids:
                    color = (0, 0, 255)

                previous_centers[track_id] = current_center

                # Рамка вокруг автомобиля
                cv2.rectangle(
                    frame,
                    (int(x1), int(y1)),
                    (int(x2), int(y2)),
                    color,
                    2
                )

                # Центр объекта
                cv2.circle(
                    frame,
                    current_center,
                    5,
                    (255, 0, 0),
                    -1
                )

                # ID объекта и уверенность модели
                label = f"ID:{track_id} car {confidence:.2f}"

                cv2.putText(
                    frame,
                    label,
                    (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

        # Линия подсчёта
        cv2.line(
            frame,
            LINE_POINT_1,
            LINE_POINT_2,
            (255, 255, 0),
            3
        )

        cv2.putText(
            frame,
            "counting line",
            (LINE_POINT_1[0], LINE_POINT_1[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2
        )

        # Счётчик машин
        draw_text_with_background(
            frame,
            f"Cars: {car_count}",
            (30, 55)
        )

        output.write(frame)

    output.release()

    print("Готово!")
    print(f"Итоговое видео сохранено: {OUTPUT_VIDEO}")
    print(f"Всего машин посчитано: {car_count}")


## 6. Просмотр результата

Если видео не отображается внутри ноутбука из-за размера файла, его можно открыть отдельно из папки проекта.


In [ ]:
display(Video(OUTPUT_VIDEO, embed=False))


## 7. Краткий вывод

В работе была создана программа для подсчёта автомобилей на видео.  
Программа находит транспортные средства, отслеживает их по ID, отображает bounding boxes, центры объектов, линию подсчёта и итоговое количество пересечений.

При тестировании результат зависит от качества видео, размера объектов на кадре и положения линии подсчёта.
